# Homography Transform

In [20]:
import os

import cv2
import numpy as np

import mon

# Arguments
image_stem = "0000135_01327_d_0000151"

# Directories and files
current_dir  = mon.Path(os.getcwd())
root_dir     = current_dir.parents[0]
data_dir     = root_dir / "data" / "bev"
image1_file  = data_dir / f"{image_stem}_1.jpg"
label1_file  = data_dir / f"{image_stem}_1.txt"
visual1_file = data_dir / f"{image_stem}_1_viz.jpg"
classes_file = data_dir / f"classes.yaml"
image2_file  = data_dir / f"{image_stem}_2.jpg"
label2_file  = data_dir / f"{image_stem}_2.txt"
visual2_file = data_dir / f"{image_stem}_2_viz.jpg"

# Load data
image1    = cv2.imread(str(image1_file))
image2    = cv2.imread(str(image2_file))
h1, w1, _ = image1.shape
h2, w2, _ = image2.shape
bs1       = mon.hbb.load(path=label1_file, fmt=mon.BBoxFormat.YOLO, imgsz=(h1, w1))
# bs2      = mon.hbb.load(path=label2_file, fmt=mon.BBoxFormat.YOLO, imgsz=(h, w))
classes   = mon.load_config(classes_file, verbose=False)
classes   = classes.get("classes", [])

pts1      = np.array([[265, 276], [670, 415], [880, 285], [695, 720], [1170, 620]], dtype=np.float32)
pts2      = np.array([[370, 159], [710, 405], [985, 130], [685, 705], [965,  650]], dtype=np.float32)
H, mask   = cv2.findHomography(pts1, pts2, cv2.RANSAC, 5.0) # H is the homography matrix
print("Homography Matrix:\n", H)

# Transform bbox
bs1_voc = mon.hbb.convert(bs1, fmt=mon.BBoxFormat.YOLO2VOC, imgsz=(h1, w1))
bs2_voc = []
for j, b in enumerate(bs1_voc):
    # b_cors  = mon.hbb.corners_pts(b[0:4]).astype(np.float32).reshape(-1, 1, 2)
    b_cors  = np.array([[b[0], b[1]], [b[2], b[1]], [b[2], b[3]], [b[0], b[3]]], dtype=np.float32).reshape(-1, 1, 2)
    b_cors_ = cv2.perspectiveTransform(b_cors, H)
    bs2_voc.append(
        [
            b_cors_[0, 0, 0],
            b_cors_[0, 0, 1],
            b_cors_[2, 0, 0],
            b_cors_[2, 0, 1],
            b[4],
        ]
    )
bs2_voc = np.array(bs2_voc)

# Visualize
viz1 = image1.copy()
for j, b in enumerate(bs1_voc):
    if len(b) >= 6:
        l = f"{j} {int(b[4])}: {b[5]:.4f}"
    else:
        l = f"{j} {int(b[4])}"
    viz1 = mon.dtypes.draw_bbox(
        image     = viz1,
        bbox      = b,
        label     = "",
        color     = classes[int(b[4])]["color"],
        thickness = 2,
        fill      = False,
    )

viz2 = image2.copy()
for j, b in enumerate(bs2_voc):
    if len(b) >= 6:
        l = f"{j} {int(b[4])}: {b[5]:.4f}"
    else:
        l = f"{j} {int(b[4])}"
    viz2 = mon.dtypes.draw_bbox(
        image     = viz2,
        bbox      = b,
        label     = "",
        color     = classes[int(b[4])]["color"],
        thickness = 2,
        fill      = False,
    )

# Save
cv2.imwrite(str(visual1_file), viz1)
cv2.imwrite(str(visual2_file), viz2)

# with open(fft_label_file, "w") as f:
#     for b in fft_bboxes:
#         f.write(f"{int(b[4])} {b[0]:.32f} {b[1]:.32f} {b[2]:.32f} {b[3]:.32f}\n")

Homography Matrix:
 [[   -0.71349     -48.912      8213.3]
 [     12.243     -70.301       13806]
 [   0.018636   -0.075135           1]]


True